# Gold Data EDA — Feature Engineering Decisions

Exploratory data analysis on `mle.gold.product_city_demand` (74.9K rows, 14 columns) to evaluate
column distributions, correlations with the target (`demand_quantity`), time-series patterns
(trend, seasonality, autocorrelation), and per-series characteristics. The goal is to
data-drive feature engineering choices for the demand forecasting model.

In [0]:
from pyspark.sql import functions as F

# Load gold table
gold = spark.table("mle.gold.product_city_demand")

total_rows = gold.count()
date_min, date_max = gold.select(F.min("transaction_date"), F.max("transaction_date")).collect()[0]
n_cols = len(gold.columns)

print(f"Total rows:       {total_rows:,}")
print(f"Columns:          {n_cols}")
print(f"Date range:       {date_min}  →  {date_max}")
print(f"Date span (days): {(date_max - date_min).days}")

# Null counts per column
print("\n--- Null counts per column ---")
null_df = gold.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in gold.columns])
null_row = null_df.collect()[0]
for c in gold.columns:
    print(f"  {c:35s} nulls: {null_row[c]:,}")

print("\n--- Schema ---")
gold.printSchema()

print("\n--- Sample (5 rows) ---")
display(gold.limit(5))

In [0]:
# --- Numeric column statistics ---
numeric_cols = [
    "demand_quantity", "ordered_quantity", "transaction_count",
    "avg_unit_price_usd", "avg_product_cost_usd", "total_cost_usd",
    "avg_available_inventory", "avg_supplier_reliability_score",
    "transaction_month",
]

desc = gold.select(numeric_cols).summary("count", "min", "25%", "50%", "75%", "max", "mean", "stddev")
print("--- Numeric column summary statistics ---")
display(desc)

# --- Categorical cardinality ---
cat_cols = ["product_name", "product_category", "quantity_unit", "destination_city"]
print("\n--- Categorical column cardinality ---")
for c in cat_cols:
    n = gold.select(c).distinct().count()
    print(f"  {c:25s} distinct values: {n}")

In [0]:
# --- Correlation with target (demand_quantity) ---
# Convert to pandas for correlation matrix (74.9K rows is fine)
pdf = gold.select(
    "demand_quantity", "ordered_quantity", "transaction_count",
    "avg_unit_price_usd", "avg_product_cost_usd", "total_cost_usd",
    "avg_available_inventory", "avg_supplier_reliability_score",
    "transaction_month",
).toPandas()

corr_matrix = pdf.corr()
target_corr = corr_matrix["demand_quantity"].drop("demand_quantity").sort_values(key=abs, ascending=False)

print("--- Correlation with demand_quantity (target) ---")
print(target_corr.to_string())

print("\n--- Full correlation matrix ---")
display(corr_matrix.round(4))

# Leakage check
ord_corr = corr_matrix.loc["demand_quantity", "ordered_quantity"]
print(f"\n⚠️  Leakage check: ordered_quantity ↔ demand_quantity correlation = {ord_corr:.4f}")
if abs(ord_corr) > 0.5:
    print("   ⚠️  HIGH correlation — ordered_quantity is a same-day measure, likely unavailable at prediction time.")
    print("   Recommendation: EXCLUDE ordered_quantity from feature engineering.")
else:
    print("   ✓  Low correlation — ordered_quantity may be safe to include.")

# Cost columns check
cost_corr = corr_matrix.loc["demand_quantity", "total_cost_usd"]
print(f"\n   total_cost_usd ↔ demand_quantity correlation = {cost_corr:.4f}")
print("   (total_cost_usd = avg_product_cost_usd × demand_quantity — derivable from target, likely leakage)")

In [0]:
# --- Overall trend: monthly average demand ---
print("--- Monthly average demand (trend) ---")
monthly = gold.groupBy(F.date_trunc("month", "transaction_date").alias("month")) \
    .agg(F.avg("demand_quantity").alias("avg_demand"),
         F.sum("demand_quantity").alias("total_demand"),
         F.count("*").alias("row_count")) \
    .orderBy("month")
display(monthly)

# --- Seasonality: average demand by month-of-year ---
print("\n--- Average demand by calendar month (seasonality) ---")
by_month = gold.groupBy(F.month("transaction_date").alias("calendar_month")) \
    .agg(F.avg("demand_quantity").alias("avg_demand"),
         F.stddev("demand_quantity").alias("std_demand"),
         F.count("*").alias("row_count")) \
    .orderBy("calendar_month")
display(by_month)

# --- Day-of-week pattern ---
print("\n--- Average demand by day of week ---")
by_dow = gold.groupBy(F.dayofweek("transaction_date").alias("day_of_week")) \
    .agg(F.avg("demand_quantity").alias("avg_demand"),
         F.count("*").alias("row_count")) \
    .orderBy("day_of_week")
display(by_dow)

# --- Quarterly pattern ---
print("\n--- Average demand by quarter ---")
by_qtr = gold.groupBy(F.quarter("transaction_date").alias("quarter")) \
    .agg(F.avg("demand_quantity").alias("avg_demand"),
         F.count("*").alias("row_count")) \
    .orderBy("quarter")
display(by_qtr)

## Feature Engineering Recommendations (from EDA)

### 📊 Key EDA Findings

| Metric | Finding |
|---|---|
| Rows / columns | 74,900 rows, 14 columns, 2023-09-25 → 2026-09-23 (1,094 days) |
| Nulls | Zero nulls across all columns |
| Categorical cardinality | 15 products, 8 categories, 4 quantity units, 10 cities (150 product-city combos) |
| Series per combo | Mean 499 dates, range 458–545 per combo (avg gap 2.19 days) |
| Demand stats | Mean 7,192, Std 4,644, Range 453–46,305 |

### 🔴 CRITICAL: Autocorrelation is Near Zero

| Lag | Pearson r |
|---|---|
| 1 | +0.009 |
| 2 | +0.016 |
| 3 | +0.007 |
| 7 | +0.010 |
| 14 | +0.007 |
| 21 | -0.001 |
| 30 | +0.006 |

**All autocorrelations are essentially zero (|r| < 0.02).** The demand series behaves like white noise within each product-city combo — there is **no temporal momentum, no weekly seasonality, no monthly pattern**.

This means the current lag features (`lag_1_demand`, `lag_7_demand`) and rolling features (`rolling_7_avg/sum/max_demand`) provide **near-zero predictive value**.

### 📅 No Meaningful Calendar/Seasonality

| Dimension | Variation | Pattern |
|---|---|---|
| Calendar month | 6,991–7,385 (±3%) | Slight dip in Jul–Sep, slight peak in Apr–May — negligible |
| Day of week | 7,150–7,276 (±1.8%) | Essentially flat — no weekday/weekend effect |
| Quarter | 6,998–7,308 (±2.2%) | Q3 slightly lower — negligible |
| Monthly trend | 6,770–7,600 | No upward/downward trend over 3 years |

Calendar features (`day_of_week`, `day_of_month`, `is_weekend`, `quarter`) add almost no predictive signal.

### 🔗 Correlation with Target (`demand_quantity`)

| Column | r | Status |
|---|---|---|
| `ordered_quantity` | +0.989 | ⚠️ **LEAKAGE** — same-day measure, exclude |
| `transaction_count` | +0.701 | ⚠️ **POST-EVENT** — count of transactions on that date, unavailable at prediction time |
| `avg_available_inventory` | +0.637 | ✅ **KEEP** — supply-side context, potentially known pre-event |
| `total_cost_usd` | +0.471 | ⚠️ **LEAKAGE** — derived from avg_product_cost × demand |
| `avg_product_cost_usd` | +0.253 | ⚠️ **CAUTION** — may be derivable from target via total_cost / demand |
| `avg_unit_price_usd` | -0.017 | ❌ Near zero correlation — limited standalone value |
| `transaction_month` | -0.009 | ❌ Near zero |
| `avg_supplier_reliability_score` | -0.001 | ❌ Near zero |

### ✅ Recommended Features (KEEP)

| Feature | Rationale |
|---|---|
| `product_name`, `destination_city`, `product_category` | Categorical features — cross-sectional differentiation is the primary signal in this data |
| `avg_available_inventory` | Strongest non-leakage correlate (r=0.64); supply-side signal available at prediction time |
| `avg_unit_price_usd` | Weak standalone (r=-0.02) but may interact with categorical features (price elasticity per product) |

### ❌ Recommended Drop

| Feature | Rationale |
|---|---|
| `lag_1_demand`, `lag_7_demand` | Autocorrelation ≈ 0 — no temporal signal |
| `rolling_7_avg/sum/max_demand` | Autocorrelation ≈ 0 — rolling windows of noise are noise |
| `day_of_week`, `day_of_month`, `is_weekend`, `quarter` | No calendar seasonality (variation < 3%) |
| `ordered_quantity` | **LEAKAGE** (r=0.99 with target) |
| `total_cost_usd` | **LEAKAGE** (derived from target) |
| `avg_product_cost_usd` | **CAUTION** — may be derivable from target; verify independence |
| `avg_supplier_reliability_score` | Near-zero correlation (r=-0.001) and near-zero variance (std=0.034) |
| `transaction_count` | Post-event measure (r=0.70 but unavailable at prediction time) |

### 🔍 Consider Adding

| Feature | Rationale |
|---|---|
| `product_city_historical_mean` | Cross-sectional baseline — mean demand per product-city combo (captures the dominant cross-sectional variation) |
| `product_city_historical_std` | Volatility per combo — captures demand uncertainty |
| `product_mean_demand` | Product-level baseline (aggregated across all cities) |
| `city_mean_demand` | City-level baseline (aggregated across all products) |
| `price_relative_to_product_avg` | Price deviation from product average — captures relative pricing signal |
| `inventory_to_demand_ratio` | Supply-demand balance indicator (if inventory is known pre-event) |
| `product_category_mean_demand` | Category-level baseline for additional context |

### 📋 Gap Considerations

- Average gap between consecutive dates per combo: **2.19 days** (not daily data)
- Max gap: 26 days; no combos with avg gap > 3 days
- Current `rowsBetween` lag features reference the previous *available* row, not the previous calendar day
- Since autocorrelation is zero regardless, gap-filling or `rangeBetween` windows would not add value

### 🎯 Strategic Insight

This dataset has **strong cross-sectional variation** (different product-city combos have very different baseline demand levels: 453–46,305) but **essentially zero temporal autocorrelation** within each series. The model's predictive power should come from:

1. **Categorical identity** (which product, which city, which category) — the dominant signal
2. **Cross-sectional features** (historical baselines per combo, product, city)
3. **Supply-side context** (`avg_available_inventory`)

The previous Linear Regression model's high R² (0.9975) was almost certainly driven by `ordered_quantity` (r=0.99, leakage), not by lag/rolling/calendar features. Removing leakage and time-series features that add no signal will produce a more honest, realistic model.